In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import os
import h5py
from temporaldata import Data

from auditorydecoding.windowing import extract_windows
from auditorydecoding.plotting import plot_signal, plot_pca_variance, plot_covariance, plot_spectrum
from scipy.signal import butter, sosfilt

/capstor/scratch/cscs/awgolab/mila_test/auditorydecoding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-06-03 14:43:19,481	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
WINDOW_LENGTH = 0.5  # seconds
FOLD_NUM = 0
SPLIT_TYPE = "intrasession-causal"
TASK_TYPE = "acoustic_stim"

DATA_ROOT = "/capstor/store/cscs/swissai/a0091/processed/neurosoft_monkeys_2026"
RECORDING_ID = "sub-02_ses-01_task-AcousStim_acq-RH_desc-raw"

In [3]:
RECORDING_IDS = ["sub-01_ses-010_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-011_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-012_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-013_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-014_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-015_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-016_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-01_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-02_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-03_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-04_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-05_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-06_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-07_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-08_task-AcousStim_acq-RH_desc-raw",
"sub-01_ses-09_task-AcousStim_acq-RH_desc-raw",
"sub-02_ses-01_task-AcousStim_acq-RH_desc-raw",
"sub-02_ses-02_task-AcousStim_acq-RH_desc-raw",
"sub-02_ses-03_task-AcousStim_acq-RH_desc-raw",
"sub-02_ses-04_task-AcousStim_acq-RH_desc-raw",
"sub-02_ses-05_task-AcousStim_acq-RH_desc-raw",
"sub-03_ses-01_task-AcousStim_acq-RH_desc-raw",
"sub-04_ses-01_task-AcousStim_acq-LH_desc-raw",
"sub-04_ses-01_task-AcousStim_acq-RH_desc-raw",
"sub-05_ses-01_task-AcousStim_acq-LH_desc-raw",
"sub-05_ses-01_task-AcousStim_acq-RH_desc-raw",
"sub-06_ses-01_task-AcousStim_acq-RH_desc-raw"]

In [4]:
path = os.path.join(DATA_ROOT, RECORDING_ID + ".h5")

with h5py.File(path) as f:
    data = Data.from_hdf5(f, lazy=False)

CHANNEL_NAMES = data.channels.id

In [5]:
keep_channels = data.channels.type == "ecog"

In [14]:
print(data.keys())

['acoustic_stim_trials', 'brainset', 'channels', 'device', 'ecog', 'on_vs_off_trials', 'session', 'splits', 'subject']


In [20]:
print(data.acoustic_stim_trials.keys())

['start', 'end', 'behavior_ids', 'behavior_labels', 'recording_id', 'timestamps']


In [23]:
print(np.mean(data.ecog.signal, axis=0).shape)

(32,)


In [ ]:
for ch in range(ecog.shape[1]):
    mu = np.mean(ecog[:, ch])

In [17]:
for data_id in RECORDING_IDS:
    path = os.path.join(DATA_ROOT, data_id + ".h5")

    with h5py.File(path) as f:
        data = Data.from_hdf5(f, lazy=False)

    keep_channels = data.channels.type == "ecog"

    ecog = data.ecog.signal[: , keep_channels]
    print(f"mean: {np.mean(ecog)} std: {np.std(ecog)} max: {np.max(ecog)} min: {np.min(ecog)}")

mean: -5.73831989011244e-06 std: 0.0012195362268044695 max: 0.01249962734375 min: -0.012500008593749999
mean: 2.9440248429518194e-05 std: 0.0008852096754944392 max: 0.01249962734375 min: -0.012500008593749999
mean: 1.607072719745878e-06 std: 0.0004138391138835467 max: 0.01249962734375 min: -0.012500008593749999
mean: 1.468054278348644e-05 std: 0.0001862231234641643 max: 0.01249962734375 min: -0.012500008593749999
mean: 1.777750971181747e-05 std: 0.0003999061589318146 max: 0.01249962734375 min: -0.012500008593749999
mean: -0.00011328757882219807 std: 0.00420268592707707 max: 0.256723975 min: -0.1650590125
mean: -7.661999985301274e-07 std: 0.0001980407877616115 max: 0.0252857390625 min: -0.00693207265625
mean: 0.00020381631976846868 std: 0.001387339983568818 max: 0.01249962734375 min: -0.012500008593749999
mean: 2.2992741649869752e-05 std: 0.0003321415799600953 max: 0.01249962734375 min: -0.0095722265625
mean: 5.7329570540191696e-05 std: 0.0013448416681570796 max: 0.01249962734375 min: -

In [20]:
rng = np.random.default_rng(42)

In [21]:
ids = data.acoustic_stim_trials.behavior_ids
unique_classes, counts = np.unique(ids, return_counts=True)

min_count = counts.min()

print(f"Class counts before balancing:")
for cls, cnt in zip(unique_classes, counts):
    print(f"  Class {cls}: {cnt} trials")
print(f"\nTarget count per class: {min_count}")

Class counts before balancing:
  Class 7: 710 trials
  Class 8: 723 trials
  Class 10: 2029 trials

Target count per class: 710


In [49]:
print(len(counts))

3


In [46]:
print(unique_classes[counts > 710])

[ 8 10]


In [44]:

cls_indices = np.where(ids == 10)[0]  
sampled = rng.choice(cls_indices, size=50, replace=False)
u, c = np.unique(sampled, return_counts=True)
print(c)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [ ]:
balanced_indices = []
for cls in unique_classes:
    cls_indices = np.where(ids == cls)[0]             
    sampled = rng.choice(cls_indices, size=min_count, replace=False)
    balanced_indices.append(sampled)

balanced_indices = np.concatenate(balanced_indices)
rng.shuffle(balanced_indices)

balanced_ids = ids[balanced_indices]

print(f"\nClass counts after balancing:")
unique_b, counts_b = np.unique(balanced_ids, return_counts=True)
for cls, cnt in zip(unique_b, counts_b):
    print(f"  Class {cls}: {cnt} trials")



Class counts after balancing:
  Class 7: 710 trials
  Class 8: 710 trials
  Class 10: 710 trials


In [19]:

unique, counts = np.unique(data.acoustic_stim_trials.behavior_ids, return_counts=True)

print(f"We have {unique} items at {counts} counts")

We have [ 7  8 10] items at [ 710  723 2029] counts
